In [9]:
import torch
import cv2
import numpy as np
import mediapipe as mp
from torchvision import transforms
from PIL import Image
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [10]:
from torchvision import models
import torch.nn as nn

Age_bucket = 20

model = models.resnet34(weights=None)

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, Age_bucket)
)

model.load_state_dict(torch.load("../models/Training_Data_Bucket_Save.pth", map_location=device))
model.to(device)
model.eval()


ipykernel_26312\2688814984.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../models/Training_Data_Bucket_Save.pth", map_location=devi

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [11]:
mp_face_detection = mp.solutions.face_detection

def preprocess_face(image_path, img_size=224):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as fd:
        results = fd.process(img_rgb)

        if not results.detections:
            print(f"No face detected: {image_path}")
            return None

        d = results.detections[0].location_data.relative_bounding_box

        h, w, _ = img_rgb.shape
        x1 = int(d.xmin * w)
        y1 = int(d.ymin * h)
        w_box = int(d.width * w)
        h_box = int(d.height * h)

        pad = int(0.25 * max(w_box, h_box)) #25% padding increase helped the for better prediction so we continued the trend for real world images too
        x1 = max(0, x1 - pad)
        y1 = max(0, y1 - pad)
        x2 = min(w, x1 + w_box + 2*pad)
        y2 = min(h, y1 + h_box + 2*pad)

        cropped = img_rgb[y1:y2, x1:x2]

        resized = cv2.resize(cropped, (img_size, img_size))
        return resized


In [12]:
transform_infer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def predict_age_bucket(model, image):
    img_tensor = transform_infer(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        pred = output.argmax(dim=1).item()

    return pred, output.softmax(dim=1)[0].cpu().numpy()


In [13]:
folder = "../real_world_images"

for file in os.listdir(folder):
    if file.lower().endswith(('.jpg', '.png', '.jpeg')):
        path = os.path.join(folder, file)
        face = preprocess_face(path)

        if face is None:
            continue

        bucket, probs = predict_age_bucket(model, face)
        print(f"{file} → Bucket: {bucket} → Age range: {bucket*6}-{bucket*6+5}")


Harsh iamge.jpeg → Bucket: 2 → Age range: 12-17
Harsh image .jpeg → Bucket: 4 → Age range: 24-29
Image age 23.jpeg → Bucket: 5 → Age range: 30-35
IMG-20230219-WA0000.jpg → Bucket: 4 → Age range: 24-29
IMG-20230417-WA0000.jpg → Bucket: 4 → Age range: 24-29
IMG-20250704-WA0028.jpg → Bucket: 4 → Age range: 24-29
IMG-20250920-WA0000.jpg → Bucket: 0 → Age range: 0-5
IMG-20251129-WA0019.jpg → Bucket: 1 → Age range: 6-11
IMG-20251129-WA0020.jpg → Bucket: 4 → Age range: 24-29
photo 1 Age26.png → Bucket: 4 → Age range: 24-29
photo 2 Age 26.jpg → Bucket: 2 → Age range: 12-17
photo 3 Age26.jpg → Bucket: 4 → Age range: 24-29
photo 4 Age26.jpg → Bucket: 3 → Age range: 18-23
photo 5 Age26.jpg → Bucket: 2 → Age range: 12-17
photo 6 Age26.jpg → Bucket: 4 → Age range: 24-29
photo 7  Age26.jpg → Bucket: 4 → Age range: 24-29
photo 8  Age26.jpg → Bucket: 3 → Age range: 18-23
Photo Age25(2).jpeg → Bucket: 2 → Age range: 12-17
Photo Age25.jpeg → Bucket: 4 → Age range: 24-29
Photo Age26.jpeg → Bucket: 8 → Ag